In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import scipy.io as scp
import numpy as np
import matplotlib.pyplot as plt

from torch_geometric.nn import GCNConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import roc_auc_score

Load the data

In [6]:
data = scp.loadmat('ACM.mat')
print(list(data.keys()))

['__header__', '__version__', '__globals__', 'Network', 'Label', 'Attributes', 'Class']


Create the Encoders

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(Encoder, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, latent_dim)

    def forward(self, x, edge):
        x = self.conv1(x, edge)
        x = torch.relu(x)

        x = self.conv2(x, edge)
        x = torch.relu(x)
        return x

class AtrDecoder(nn.Module):
    """
    Attribute decoder for graph
    """
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super(AtrDecoder, self).__init__()
        self.conv1 = GCNConv(latent_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)
    
    def forward(self, z, edge):
        x = self.conv1(z, edge)
        x = torch.relu(x)
        
        x = self.conv2(x, edge)
        return x

class Decoder(nn.Module):
    """
    Z @ Z^T reconstruction of the adjacency matrix.
    """
    def __init__(self, latent_dim):
        super(Decoder, self).__init__()
        self.conv = GCNConv(latent_dim, latent_dim)
    
    def forward(self, z, edge):

        z = self.conv(z, edge)
        z = torch.relu(z)
        
        A_hat = torch.matmul(z, z.t())
        
        return A_hat


class GraphAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(GraphAutoencoder, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.atr_dec = AtrDecoder(latent_dim, hidden_dim, input_dim)
        self.dec = Decoder(latent_dim)
    
    def forward(self, x, edge_index):
        # Encode
        z = self.encoder(x, edge_index)
        
        # Decode attributes
        x_hat = self.atr_dec(z, edge_index)
        
        # Decode
        A_hat = self.dec(z, edge_index)
        
        return x_hat, A_hat
    
def loss(X, x_hat, A, A_hat, alpha):
    # Frobenius norm squared 
    attr_loss = torch.norm(X - x_hat, p="fro") ** 2
    struct_loss = torch.norm(A - A_hat, p="fro") ** 2

    # Combined loss
    total_loss = alpha * attr_loss + (1 - alpha) * struct_loss

    return total_loss, attr_loss, struct_loss